# Final PBFAIR analysis

This notebook reproduces only the analyses reported in the paper:

1. comparison of methods providing guarantees;
2. empirical minority-class F1–fairness trade-off;
3. impact of the deterministic-conversion heuristics.

Protocol:

- seven datasets and five seeds (`541944` to `541948`);
- PAC-Bayes selection using the equally weighted mean of the risk and fairness bounds;
- FERM and Reduction selection on validation data;
- no test metric used for configuration selection;
- PAC-Bayes F1 reconstructed seed by seed from class-conditional errors.


## 1. Imports and paths

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import sys

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FormatStrFormatter
import numpy as np
import pandas as pd
from IPython.display import Code, Markdown, display


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (
            (candidate / "results").is_dir()
            and (candidate / "analysis").is_dir()
            and (candidate / "data" / "init.py").is_file()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not find results/, analysis/, and data/init.py."
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RESULTS_ROOT = REPO_ROOT / "results"
DATA_ROOT = REPO_ROOT / "data"
EXPORT_ROOT = REPO_ROOT / "analysis" / "exports"

print(f"Repository root: {REPO_ROOT}")


## 2. Protocol configuration

The constants below match the published protocol.


In [ ]:
DATASETS = (
    "ACSINC",
    "ACSMOB",
    "ACSTRAVEL",
    "COMPAS",
    "GERMAN",
    "LAW",
    "MEPS",
)
PAPER_SEEDS = tuple(range(541944, 541949))
TRIPLE_ALPHA_VALUES = (0.1, 0.25, 0.5, 0.75, 0.9)
ONETO_TRADEOFF_VALUES = (0.1, 1.0, 10.0, 100.0)
POSTERIOR_FAMILIES = ("categorical", "dirichlet", "gaussian")

FAIRNESS_METRICS = ("DP", "EO", "EP")
SENSITIVE_VARIABLES = ("GENDER", "OTHER")
PLOT_FAIRNESS_METRICS = FAIRNESS_METRICS

EMPIRICAL_Y_METRIC = "fscore"
SHOW_EMPIRICAL_IDEAL_ARROW = True

BOUND_METHODS = (
    "ours_det_rf",
    "ours_sto_rf",
    "triple_oneto_rf",
    "oneto_rf",
)

EMPIRICAL_METHODS = (
    "ours_det_rf",
    "ours_sto_rf",
    "ferm_linear",
    "reduction_trees",
)

F1_ZERO_TOLERANCE = 1e-12
CONSTANT_TOLERANCE = 1e-7

SAVE_EXPORTS = True
EXPORT_TABLES = False
EXPORT_LATEX_TABLES = True
OUTPUT_DIR = EXPORT_ROOT
EXPORT_PDF_PAD_INCHES = 0.03

PAPER_STYLE = {
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "grid.color": "#D9D9D9",
    "grid.linewidth": 0.6,
    "grid.alpha": 0.45,
    "figure.dpi": 120,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.bbox": "tight",
    "savefig.pad_inches": EXPORT_PDF_PAD_INCHES,
    "savefig.facecolor": "white",
    "savefig.transparent": False,
}
plt.rcParams.update(PAPER_STYLE)

if len(DATASETS) != len(set(DATASETS)):
    raise ValueError("DATASETS contains duplicates.")
if len(PAPER_SEEDS) != 5:
    raise ValueError("The protocol must contain exactly five seeds.")
if not set(PLOT_FAIRNESS_METRICS).issubset(FAIRNESS_METRICS):
    raise ValueError(
        "PLOT_FAIRNESS_METRICS must be a subset of FAIRNESS_METRICS."
    )
if EMPIRICAL_Y_METRIC != "fscore":
    raise ValueError("The paper uses F1 for the empirical fairness trade-off.")

FAIRNESS_DISPLAY_NAMES = {
    "EP": "EOP",
}


def fairness_display(fairness_metric):
    return FAIRNESS_DISPLAY_NAMES.get(
        fairness_metric,
        fairness_metric,
    )


print(f"Datasets: {', '.join(DATASETS)}")
print(f"Seeds: {PAPER_SEEDS[0]}–{PAPER_SEEDS[-1]}")


## 3. Compared methods

In [ ]:
SYSTEM_SPECS = {
    "ours_det_rf": {
        "label": "Ours - det.",
        "family": "pac_bayes",
        "risk": "Triple",
        "fairness_bound": "ours_stochastic_fairness",
        "predictor": "RandomForests",
        "estimator": "det",
        "phase": "af",
        "risk_bound_metric": "det_OA_bnd_up_af",
        "fairness_bound_metric": "det_{fairness_metric}_bnd_up_af",
        "supported_metrics": ("DP", "EO", "EP"),
    },
    "ours_sto_rf": {
        "label": "Ours - sto.",
        "family": "pac_bayes",
        "risk": "Triple",
        "fairness_bound": "ours_stochastic_fairness",
        "predictor": "RandomForests",
        "estimator": "gbs",
        "phase": "bf",
        "risk_bound_metric": "gbs_OA_bnd_up_bf",
        "fairness_bound_metric": "gbs_{fairness_metric}_bnd_up_bf",
        "supported_metrics": ("DP", "EO", "EP"),
    },
    "triple_oneto_rf": {
        "label": "O1 - sto.",
        "family": "pac_bayes",
        "risk": "Triple",
        "fairness_bound": "oneto_fairness",
        "predictor": "RandomForests",
        "estimator": "gbs",
        "phase": "af",
        "risk_bound_metric": "gbs_OA_bnd_up_af",
        "fairness_bound_metric": "gbs_{fairness_metric}_bnd_ou_af",
        "supported_metrics": ("DP", "EO", "EP"),
    },
    "oneto_rf": {
        "label": "O2 - sto.",
        "family": "pac_bayes",
        "risk": "Oneto",
        "fairness_bound": "oneto_fairness",
        "predictor": "RandomForests",
        "estimator": "gbs",
        "phase": "af",
        "risk_bound_metric": "gbs_OA_bnd_up_af",
        "fairness_bound_metric": "gbs_{fairness_metric}_bnd_up_af",
        "supported_metrics": ("DP", "EP"),
    },
    "ferm_linear": {
        "label": "FERM",
        "family": "ferm",
        "risk": "FERM",
        "fairness_bound": "ferm",
        "predictor": "LinearFERM",
        "estimator": "det",
        "phase": "af",
        "supported_metrics": ("EP",),
    },
    "reduction_trees": {
        "label": "Reduction",
        "family": "reduction",
        "risk": "Agarwal",
        "fairness_bound": "agarwal",
        "predictor": "CART",
        "estimator": "gibbs",
        "phase": "af",
        "supported_metrics": ("DP", "EO", "EP"),
    },
}

ACTIVE_SYSTEMS = tuple(
    dict.fromkeys((*BOUND_METHODS, *EMPIRICAL_METHODS))
)
unknown_systems = set(ACTIVE_SYSTEMS) - set(SYSTEM_SPECS)
extra_systems = set(SYSTEM_SPECS) - set(ACTIVE_SYSTEMS)
if unknown_systems:
    raise ValueError(f"Unknown systems: {sorted(unknown_systems)}")
if extra_systems:
    raise ValueError(
        f"SYSTEM_SPECS contains methods outside the paper: {sorted(extra_systems)}"
    )

SYSTEM_LABELS = {
    system: SYSTEM_SPECS[system]["label"] for system in SYSTEM_SPECS
}
SYSTEM_COLORS = {
    "ours_det_rf": "#56B4E9",
    "ours_sto_rf": "#0072B2",
    "triple_oneto_rf": "#E69F00",
    "oneto_rf": "#F0E442",
    "ferm_linear": "#CC79A7",
    "reduction_trees": "#009E73",
}
SYSTEM_HATCHES = {
    "ours_det_rf": "///",
    "ours_sto_rf": "...",
    "triple_oneto_rf": "\\\\",
    "oneto_rf": "oo",
    "ferm_linear": "**",
    "reduction_trees": "xx",
}

TASK_KEYS = ["dataset", "fairness_metric", "sensible_variable"]
PAIR_KEYS = [*TASK_KEYS, "seed"]


## 4. Loading results

In [ ]:
def to_float(value):
    if value is None:
        return np.nan
    try:
        return float(value)
    except (TypeError, ValueError):
        return float(value.item())


def load_result_dict(path):
    return np.load(path, allow_pickle=True).item()


def parse_config(path):
    parts = path.parent.relative_to(RESULTS_ROOT).parts
    if len(parts) < 4 or not parts[2].startswith("fairness="):
        return None

    record = {
        "dataset": parts[0],
        "risk": parts[1],
        "fairness_bound": parts[2].split("=", 1)[1],
        "predictor": parts[3],
        "config_dir": str(path.parent),
        "config_id": str(path.parent.relative_to(RESULTS_ROOT)),
    }
    for part in parts[4:]:
        if "=" in part:
            key, value = part.split("=", 1)
            record[key.replace("-", "_")] = value
    return record


def record_matches_spec(record, spec):
    return (
        record["risk"] == spec["risk"]
        and record["fairness_bound"] == spec["fairness_bound"]
        and record["predictor"] == spec["predictor"]
    )


def record_is_relevant(record):
    return any(
        record_matches_spec(record, SYSTEM_SPECS[system])
        for system in ACTIVE_SYSTEMS
    )


available_datasets = tuple(
    sorted(path.name for path in RESULTS_ROOT.iterdir() if path.is_dir())
)
missing_datasets = set(DATASETS) - set(available_datasets)
if missing_datasets:
    raise ValueError(
        "Datasets missing from results/: "
        f"{sorted(missing_datasets)}. "
        f"Available: {list(available_datasets)}"
    )

records = []
for path in sorted(RESULTS_ROOT.rglob("err-b.npy")):
    relative_parts = path.relative_to(RESULTS_ROOT).parts
    if any(part.startswith("seed=") for part in relative_parts):
        continue

    record = parse_config(path)
    if record is None:
        continue
    if record["dataset"] not in DATASETS:
        continue
    if record.get("fairness_metric") not in FAIRNESS_METRICS:
        continue
    if record.get("sensible_variable") not in SENSITIVE_VARIABLES:
        continue
    if not record_is_relevant(record):
        continue
    records.append(record)

if not records:
    raise ValueError("No result matches the configured protocol.")

candidate_configs = pd.DataFrame(
    [
        {
            key: record.get(key)
            for key in (
                "config_id",
                "dataset",
                "risk",
                "fairness_bound",
                "predictor",
                "fairness_metric",
                "sensible_variable",
                "alpha",
                "gamma",
                "distribution",
                "C",
                "config_dir",
            )
        }
        for record in records
    ]
)

print(f"Candidate configurations: {len(records):,}")
display(
    candidate_configs.groupby(
        ["risk", "fairness_bound", "predictor"],
        dropna=False,
    )
    .size()
    .rename("configurations")
    .to_frame()
)


## 5. Exact reconstruction of minority-class F1

For PAC-Bayes methods, any stored F1 value is intentionally ignored.

F1 is recomputed for every seed from:

- the class-conditional errors;
- the train/test class counts;
- the minority class in the training split, used as the positive class.

Datasets and seed files are cached and loaded only once.

In [ ]:
def fscore_from_errors(
    error_lz,
    error_lo,
    size_lz,
    size_lo,
    positive_label,
):
    if positive_label == -1:
        positive_size = size_lz
        false_positive = to_float(error_lo) * size_lo
        false_negative = to_float(error_lz) * size_lz
    else:
        positive_size = size_lo
        false_positive = to_float(error_lz) * size_lz
        false_negative = to_float(error_lo) * size_lo

    true_positive = positive_size - false_negative
    denominator = 2 * true_positive + false_positive + false_negative
    if np.isclose(denominator, 0.0):
        return np.nan
    return float(2 * true_positive / denominator)


class_size_cache = {}
seed_result_cache = {}


def load_seed_result(path):
    path = str(path)
    if path not in seed_result_cache:
        seed_result_cache[path] = load_result_dict(Path(path))
    return seed_result_cache[path]


def get_class_sizes(record, seed):
    key = (
        record["dataset"],
        record["sensible_variable"],
        record["predictor"],
        int(seed),
    )
    if key not in class_size_cache:
        from data.init import Dataset

        data = Dataset(
            dataset=record["dataset"],
            sensible_variable=record["sensible_variable"],
            pred=record["predictor"],
            data_path=DATA_ROOT,
            normalize=False,
            seed=int(seed),
            offline=True,
        )
        train_sizes = (
            len(data.y["train_0"]),
            len(data.y["train_1"]),
        )
        class_size_cache[key] = {
            "positive_label": -1 if train_sizes[0] < train_sizes[1] else 1,
            "tr": train_sizes,
            "te": (
                len(data.y["test_0"]),
                len(data.y["test_1"]),
            ),
        }
    return class_size_cache[key]


def available_seed_paths(record):
    paths_by_seed = {}
    for path in Path(record["config_dir"]).glob("seed=*/err-b.npy"):
        seed = int(path.parent.name.split("=", 1)[1])
        if seed not in PAPER_SEEDS:
            continue
        if seed in paths_by_seed:
            raise AssertionError(f"Duplicate seed: {seed}")
        paths_by_seed[seed] = path

    missing = set(PAPER_SEEDS) - set(paths_by_seed)
    if missing:
        raise FileNotFoundError(
            f"Missing seeds for {record['config_id']}: {sorted(missing)}"
        )
    return [paths_by_seed[seed] for seed in PAPER_SEEDS]


def config_seed_metric_mean(record, metric_name):
    values = [
        load_seed_result(path).get(metric_name)
        for path in available_seed_paths(record)
    ]
    if any(value is None for value in values):
        return np.nan
    values = np.asarray([to_float(value) for value in values])
    return float(np.mean(values)) if np.isfinite(values).all() else np.nan


def pac_bayes_seed_fscore(record, seed_result, split):
    if split not in {"tr", "te"}:
        return np.nan

    estimator = record["estimator"]
    phase = record["phase"]
    seed = int(seed_result["seed"])
    sizes = get_class_sizes(record, seed)
    prefix = f"{estimator}_OA_err_{split}_{phase}"
    error_lz = seed_result.get(f"{prefix}_lz")
    error_lo = seed_result.get(f"{prefix}_lo")
    if error_lz is None or error_lo is None:
        raise KeyError(
            f"Missing class-conditional errors for {record['config_id']}"
        )

    size_lz, size_lo = sizes[split]
    return fscore_from_errors(
        error_lz,
        error_lo,
        size_lz,
        size_lo,
        sizes["positive_label"],
    )


def external_seed_fscore(record, seed_result, split):
    metric = (
        f"{record['estimator']}_OA_fscore_"
        f"{split}_{record['phase']}"
    )
    value = seed_result.get(metric)
    if value is None:
        return np.nan
    return to_float(value)


def seed_fscore(record, seed_result, split):
    if record["family"] == "pac_bayes":
        return pac_bayes_seed_fscore(record, seed_result, split)
    return external_seed_fscore(record, seed_result, split)


def positive_label(record, seed_result):
    if record["family"] != "pac_bayes":
        value = seed_result.get("fscore_positive_label")
        return np.nan if value is None else int(to_float(value))
    return int(
        get_class_sizes(record, int(seed_result["seed"]))[
            "positive_label"
        ]
    )


def is_constant_classifier(record, seed_result, split="te"):
    prefix = (
        f"{record['estimator']}_OA_err_"
        f"{split}_{record['phase']}"
    )
    error_lz = seed_result.get(f"{prefix}_lz")
    error_lo = seed_result.get(f"{prefix}_lo")
    if error_lz is None or error_lo is None:
        return False

    error_lz = to_float(error_lz)
    error_lo = to_float(error_lo)
    return bool(
        (
            abs(error_lz) <= CONSTANT_TOLERANCE
            and abs(error_lo - 1.0) <= CONSTANT_TOLERANCE
        )
        or (
            abs(error_lz - 1.0) <= CONSTANT_TOLERANCE
            and abs(error_lo) <= CONSTANT_TOLERANCE
        )
    )


## 6. Configuration selection

In [ ]:
def format_metric(template, fairness_metric):
    return template.format(fairness_metric=fairness_metric)


def pb_bound_metrics(spec, fairness_metric):
    return (
        spec["risk_bound_metric"],
        format_metric(
            spec["fairness_bound_metric"],
            fairness_metric,
        ),
    )


def equal_risk_fairness_score(risk, fairness):
    if not (np.isfinite(risk) and np.isfinite(fairness)):
        return np.nan
    return 0.5 * risk + 0.5 * fairness


def pac_bayes_record_in_search_space(record, spec):
    distribution = record.get("distribution")
    if record.get("alpha") is None:
        return False
    alpha = float(record["alpha"])

    if spec["risk"] == "Oneto":
        if record.get("gamma") is None:
            return False
        gamma = float(record["gamma"])
        return (
            distribution == "categorical"
            and any(
                np.isclose(alpha, value)
                for value in ONETO_TRADEOFF_VALUES
            )
            and any(
                np.isclose(gamma, value)
                for value in ONETO_TRADEOFF_VALUES
            )
        )

    return (
        distribution in POSTERIOR_FAMILIES
        and any(
            np.isclose(alpha, value)
            for value in TRIPLE_ALPHA_VALUES
        )
    )


def pb_candidate(record, system):
    spec = SYSTEM_SPECS[system]
    fairness_metric = record["fairness_metric"]
    risk_metric, fairness_metric_name = pb_bound_metrics(
        spec,
        fairness_metric,
    )
    risk = config_seed_metric_mean(record, risk_metric)
    fairness = config_seed_metric_mean(record, fairness_metric_name)
    score = equal_risk_fairness_score(risk, fairness)
    if not np.isfinite(score):
        return None

    candidate = dict(record)
    candidate.update(spec)
    candidate.update(
        {
            "system": system,
            "selection_source": "bounds",
            "selection_score": float(score),
            "selection_risk_metric": risk_metric,
            "selection_fairness_metric": fairness_metric_name,
            "selection_risk": risk,
            "selection_fairness": fairness,
            "risk_bound_metric": risk_metric,
            "fairness_bound_metric": fairness_metric_name,
        }
    )
    return candidate


def select_pac_bayes(records, system):
    spec = SYSTEM_SPECS[system]
    selected = {}
    for record in records:
        if not record_matches_spec(record, spec):
            continue
        if record["fairness_metric"] not in spec["supported_metrics"]:
            continue
        if not pac_bayes_record_in_search_space(record, spec):
            continue

        candidate = pb_candidate(record, system)
        if candidate is None:
            continue
        key = tuple(record[field] for field in TASK_KEYS)
        tie_break = (
            candidate["selection_score"],
            candidate["config_id"],
        )
        current = selected.get(key)
        if current is None or tie_break < current[0]:
            selected[key] = (tie_break, candidate)
    return [value[1] for value in selected.values()]


def select_ferm(records, system):
    spec = SYSTEM_SPECS[system]
    selected = {}
    for record in records:
        if not record_matches_spec(record, spec):
            continue
        if record["fairness_metric"] != "EP":
            continue

        risk_metric = "det_OA_err_va_af"
        fairness_metric_name = "det_EP_err_va_af"
        risk = config_seed_metric_mean(record, risk_metric)
        fairness = config_seed_metric_mean(record, fairness_metric_name)
        score = equal_risk_fairness_score(risk, fairness)
        if not np.isfinite(score):
            continue

        candidate = dict(record)
        candidate.update(spec)
        candidate.update(
            {
                "system": system,
                "selection_source": "empirical_validation",
                "selection_score": float(score),
                "selection_risk_metric": risk_metric,
                "selection_fairness_metric": fairness_metric_name,
                "selection_risk": risk,
                "selection_fairness": fairness,
                "risk_bound_metric": None,
                "fairness_bound_metric": None,
                "C": float(record["C"]),
            }
        )
        key = tuple(record[field] for field in TASK_KEYS)
        tie_break = (
            candidate["selection_score"],
            candidate["C"],
            candidate["config_id"],
        )
        current = selected.get(key)
        if current is None or tie_break < current[0]:
            selected[key] = (tie_break, candidate)
    return [value[1] for value in selected.values()]


def select_reduction(records, system):
    spec = SYSTEM_SPECS[system]
    selected = {}
    for record in records:
        if not record_matches_spec(record, spec):
            continue
        if record["fairness_metric"] not in spec["supported_metrics"]:
            continue

        fairness_metric = record["fairness_metric"]
        risk_metric = "gibbs_OA_err_va_af"
        fairness_metric_name = f"gibbs_{fairness_metric}_err_va_af"
        risk = config_seed_metric_mean(record, risk_metric)
        fairness = config_seed_metric_mean(record, fairness_metric_name)
        score = equal_risk_fairness_score(risk, fairness)
        if not np.isfinite(score):
            continue

        candidate = dict(record)
        candidate.update(spec)
        candidate.update(
            {
                "system": system,
                "selection_source": "empirical_validation",
                "selection_score": float(score),
                "selection_risk_metric": risk_metric,
                "selection_fairness_metric": fairness_metric_name,
                "selection_risk": risk,
                "selection_fairness": fairness,
                "risk_bound_metric": None,
                "fairness_bound_metric": None,
            }
        )
        key = tuple(record[field] for field in TASK_KEYS)
        tie_break = (score, candidate["config_id"])
        current = selected.get(key)
        if current is None or tie_break < current[0]:
            selected[key] = (tie_break, candidate)
    return [value[1] for value in selected.values()]


def select_system(records, system):
    family = SYSTEM_SPECS[system]["family"]
    if family == "pac_bayes":
        return select_pac_bayes(records, system)
    if family == "ferm":
        return select_ferm(records, system)
    if family == "reduction":
        return select_reduction(records, system)
    raise ValueError(f"Unknown method family: {family}")


selection_records = []
for system in ACTIVE_SYSTEMS:
    selection_records.extend(select_system(records, system))

selection_records = sorted(
    selection_records,
    key=lambda row: (
        ACTIVE_SYSTEMS.index(row["system"]),
        row["dataset"],
        row["fairness_metric"],
        row["sensible_variable"],
    ),
)

selected_configs = pd.DataFrame(
    [
        {
            "system": row["system"],
            "system_label": row["label"],
            "family": row["family"],
            "dataset": row["dataset"],
            "fairness_metric": row["fairness_metric"],
            "sensible_variable": row["sensible_variable"],
            "predictor": row["predictor"],
            "distribution": row.get("distribution"),
            "alpha": (
                float(row["alpha"])
                if row.get("alpha") is not None
                else np.nan
            ),
            "gamma": (
                float(row["gamma"])
                if row.get("gamma") is not None
                else np.nan
            ),
            "C": (
                float(row["C"])
                if row.get("C") is not None
                else np.nan
            ),
            "selection_source": row["selection_source"],
            "selection_score": row["selection_score"],
            "selection_risk_metric": row["selection_risk_metric"],
            "selection_fairness_metric": row[
                "selection_fairness_metric"
            ],
            "config_id": row["config_id"],
            "config_dir": row["config_dir"],
        }
        for row in selection_records
    ]
)

display(
    selected_configs.groupby(
        ["system_label", "selection_source"],
        dropna=False,
    )
    .size()
    .rename("configurations")
    .to_frame()
)


## 7. Per-seed metrics and canonical tables

In [ ]:
def seed_metric(seed_result, metric):
    if metric is None:
        return np.nan
    value = seed_result.get(metric)
    return np.nan if value is None else to_float(value)


def seed_bound(seed_result, metric, record):
    if metric is None:
        return np.nan
    value = seed_metric(seed_result, metric)
    if not np.isfinite(value):
        raise AssertionError(
            f"Missing per-seed bound: {metric} in {record['config_id']}"
        )
    return value


seed_rows = []
for record in selection_records:
    for seed_path in available_seed_paths(record):
        seed_result = load_seed_result(seed_path)
        seed = int(seed_result["seed"])
        estimator = record["estimator"]
        phase = record["phase"]
        fairness_metric = record["fairness_metric"]

        row = {
            "system": record["system"],
            "system_label": record["label"],
            "family": record["family"],
            "dataset": record["dataset"],
            "fairness_metric": fairness_metric,
            "sensible_variable": record["sensible_variable"],
            "predictor": record["predictor"],
            "seed": seed,
            "config_id": record["config_id"],
            "selection_source": record["selection_source"],
            "risk_bound": seed_bound(
                seed_result,
                record.get("risk_bound_metric"),
                record,
            ),
            "fairness_bound": seed_bound(
                seed_result,
                record.get("fairness_bound_metric"),
                record,
            ),
            "positive_label": positive_label(record, seed_result),
            "f1_source": (
                "reconstructed_from_class_errors"
                if record["family"] == "pac_bayes"
                else "stored_empirical"
            ),
        }

        for split, name in (
            ("tr", "train"),
            ("va", "validation"),
            ("te", "test"),
        ):
            row[f"risk_{name}"] = seed_metric(
                seed_result,
                f"{estimator}_OA_err_{split}_{phase}",
            )
            row[f"fairness_{name}"] = seed_metric(
                seed_result,
                (
                    f"{estimator}_{fairness_metric}_"
                    f"err_{split}_{phase}"
                ),
            )
            row[f"f1_{name}"] = seed_fscore(
                record,
                seed_result,
                split,
            )

        row["zero_f1"] = bool(
            np.isfinite(row["f1_test"])
            and row["f1_test"] <= F1_ZERO_TOLERANCE
        )
        row["constant_classifier"] = is_constant_classifier(
            record,
            seed_result,
        )
        seed_rows.append(row)

selected_seeds = pd.DataFrame(seed_rows)

task_summary = (
    selected_seeds.groupby(
        ["system", "system_label", "family", *TASK_KEYS],
        as_index=False,
    )
    .agg(
        risk_bound=("risk_bound", "mean"),
        fairness_bound=("fairness_bound", "mean"),
        risk_test=("risk_test", "mean"),
        fairness_test=("fairness_test", "mean"),
        f1_test=("f1_test", "mean"),
        risk_train=("risk_train", "mean"),
        fairness_train=("fairness_train", "mean"),
        f1_train=("f1_train", "mean"),
        n_seeds=("seed", "nunique"),
        zero_f1_any_seed=("zero_f1", "any"),
        zero_f1_all_seeds=("zero_f1", "all"),
        constant_any_seed=("constant_classifier", "any"),
        constant_all_seeds=("constant_classifier", "all"),
        config_id=("config_id", "first"),
    )
    .sort_values(["system", *TASK_KEYS])
    .reset_index(drop=True)
)

display(
    task_summary.groupby(
        ["system_label", "fairness_metric"],
        as_index=False,
    )
    .agg(
        datasets=("dataset", "nunique"),
        tasks=("config_id", "size"),
        f1_test=("f1_test", "mean"),
    )
    .assign(
        fairness_metric=lambda data: (
            data["fairness_metric"].map(fairness_display)
        )
    )
)


## 8. Comparison of bound-producing methods

Each figure follows the layout used in the paper:

- one panel per dataset;
- two bar groups: Risk and Fairness;
- hatched bar: bound;
- solid bar: empirical test error.

Separate figures are generated for each fairness metric and sensitive attribute.

In [ ]:
BOUND_COLOR = "steelblue"
TEST_COLOR = "orange"


def clean_axis(axis):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.grid(axis="y", alpha=0.25, linewidth=0.7)


def sensitive_display(dataset, sensible_variable):
    if sensible_variable == "GENDER":
        return "GENDER"
    if dataset == "GERMAN" and sensible_variable == "OTHER":
        return "AGE"
    if sensible_variable == "OTHER":
        return "RACE"
    return sensible_variable


def get_paper_style_table(
    fairness_metric=None,
    sensible_variable=None,
    datasets=None,
    methods=None,
):
    if methods is None:
        methods = BOUND_METHODS

    data = task_summary[
        task_summary["risk_bound"].notna()
        & task_summary["fairness_bound"].notna()
    ].copy()

    filters = {
        "fairness_metric": fairness_metric,
        "sensible_variable": sensible_variable,
        "dataset": datasets,
        "system": methods,
    }
    for column, value in filters.items():
        if value is None:
            continue
        values = (value,) if isinstance(value, str) else tuple(value)
        data = data[data[column].isin(values)]

    data["method"] = data["system"]
    data["method_label"] = data["system_label"]
    data["panel_label"] = [
        f"{dataset} - {sensitive_display(dataset, sensible)}"
        for dataset, sensible in zip(
            data["dataset"],
            data["sensible_variable"],
        )
    ]
    return data.reset_index(drop=True)


def plot_paper_style_grid(dataframe):
    plot_data = dataframe.replace(
        [np.inf, -np.inf],
        np.nan,
    ).dropna(
        subset=[
            "risk_bound",
            "risk_test",
            "fairness_bound",
            "fairness_test",
        ],
        how="all",
    )
    if plot_data.empty:
        raise ValueError("No row remains after filtering.")

    figures = []
    for (fairness_metric, sensible_variable), group in plot_data.groupby(
        ["fairness_metric", "sensible_variable"],
        sort=False,
    ):
        datasets = [
            dataset for dataset in DATASETS
            if dataset in set(group["dataset"])
        ]
        figure, axes = plt.subplots(
            1,
            len(datasets),
            figsize=(1.35 * len(datasets), 2.45 * 0.75),
            sharey=True,
            squeeze=False,
        )
        axes = axes.ravel()
        category_positions = np.arange(2)

        plotted_values = group[
            ["risk_bound", "fairness_bound", "risk_test", "fairness_test"]
        ].to_numpy(dtype=float)
        finite_values = plotted_values[np.isfinite(plotted_values)]
        plot_upper = (
            max(
                0.75,
                0.25 * math.ceil(float(finite_values.max()) / 0.25),
            )
            if finite_values.size
            else 0.75
        )

        for axis, dataset in zip(axes, datasets):
            axis.set_box_aspect(0.75)
            panel = group[group["dataset"] == dataset]
            method_order = [
                system for system in BOUND_METHODS
                if system in set(panel["system"])
            ]
            bar_width = min(0.21, 0.8 / max(1, len(method_order)))
            center = (len(method_order) - 1) / 2

            for method_index, system in enumerate(method_order):
                row = panel[panel["system"] == system]
                if row.empty:
                    continue
                row = row.iloc[0]
                offset = (method_index - center) * bar_width
                x_values = category_positions + offset

                axis.bar(
                    x_values,
                    [row["risk_bound"], row["fairness_bound"]],
                    bar_width,
                    color=BOUND_COLOR,
                    edgecolor="black",
                    linewidth=0.7,
                    hatch=SYSTEM_HATCHES[system],
                )
                axis.bar(
                    x_values,
                    [row["risk_test"], row["fairness_test"]],
                    bar_width,
                    color=TEST_COLOR,
                    edgecolor="black",
                    linewidth=0.7,
                    hatch=SYSTEM_HATCHES[system],
                )

            axis.set_title(
                panel["panel_label"].iloc[0],
                fontsize=7,
                fontweight="bold",
            )
            axis.set_xticks(category_positions)
            axis.set_xticklabels(
                ["Risk", fairness_display(fairness_metric)],
                fontsize=9,
            )
            axis.set_ylim(0.0, plot_upper)
            axis.set_yticks(np.arange(0.0, plot_upper + 0.125, 0.25))
            axis.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
            axis.set_axisbelow(True)
            axis.grid(
                axis="y",
                color="#D9D9D9",
                linestyle="-",
                alpha=0.75,
                linewidth=0.6,
            )
            axis.tick_params(
                axis="y",
                which="major",
                left=True,
                right=True,
                direction="inout",
                length=3,
                width=0.7,
            )
            for spine in axis.spines.values():
                spine.set_visible(True)
                spine.set_color("#4D4D4D")
                spine.set_linewidth(0.8)

        method_handles = [
            Patch(
                facecolor="white",
                edgecolor="black",
                linewidth=0.7,
                hatch=SYSTEM_HATCHES[system],
                label=SYSTEM_LABELS[system],
            )
            for system in BOUND_METHODS
            if system in set(group["system"])
        ]
        legend_handles = [
            Patch(facecolor=BOUND_COLOR, edgecolor="black", label="Bound"),
            Patch(facecolor=TEST_COLOR, edgecolor="black", label="Test error"),
            *method_handles,
        ]
        figure.legend(
            handles=legend_handles,
            bbox_to_anchor=(0.01, 0.95),
            loc="upper left",
            ncol=len(legend_handles),
            frameon=False,
            fontsize=8,
            borderaxespad=0.0,
            handlelength=2.0,
            handleheight=1.0,
            handletextpad=0.45,
            columnspacing=1.1,
        )
        figure.tight_layout(rect=(0.0, 0.0, 1.0, 0.88))
        figure.subplots_adjust(wspace=0)
        figures.append(figure)
        plt.show()

    return figures


bound_artifacts = {}
for fairness_metric in PLOT_FAIRNESS_METRICS:
    for sensible_variable in SENSITIVE_VARIABLES:
        data = get_paper_style_table(
            fairness_metric=fairness_metric,
            sensible_variable=sensible_variable,
            datasets=DATASETS,
            methods=BOUND_METHODS,
        )
        figures = plot_paper_style_grid(data)
        bound_artifacts[(fairness_metric, sensible_variable)] = (
            figures[0],
            data,
        )


## 9. Impact of the deterministic heuristic

The figures compare metrics before (`bf`) and after (`af`) applying the
heuristic to the same selected configuration. Points are averages over the
available seeds. All quantities are minimized.

In [ ]:
HEURISTIC_SYSTEM = "ours_det_rf"
HEURISTIC_METRICS = (
    "risk_test",
    "fairness_test",
    "risk_bound",
    "fairness_bound",
)

HEURISTIC_COLORS = {
    "before": "#9AA0A6",
    "improved": "#009E73",
    "degraded": "#D55E00",
    "unchanged": "#6C757D",
}
HEURISTIC_LINESTYLES = {
    "improved": "-",
    "degraded": "--",
    "unchanged": ":",
}


def build_heuristic_seed_metrics(
    records,
    system=HEURISTIC_SYSTEM,
):
    rows = []
    for record in records:
        if record["system"] != system:
            continue
        if record["estimator"] != "det":
            raise AssertionError(
                f"{system} must use the deterministic estimator."
            )

        fairness_metric = record["fairness_metric"]
        metric_names = {
            "risk_test": (
                "det_OA_err_te_bf",
                "det_OA_err_te_af",
            ),
            "fairness_test": (
                f"det_{fairness_metric}_err_te_bf",
                f"det_{fairness_metric}_err_te_af",
            ),
            "risk_bound": (
                "det_OA_bnd_up_bf",
                "det_OA_bnd_up_af",
            ),
            "fairness_bound": (
                f"det_{fairness_metric}_bnd_up_bf",
                f"det_{fairness_metric}_bnd_up_af",
            ),
        }

        for seed_path in available_seed_paths(record):
            seed_result = load_seed_result(seed_path)
            row = {
                "system": system,
                "dataset": record["dataset"],
                "fairness_metric": fairness_metric,
                "sensible_variable": record["sensible_variable"],
                "predictor": record["predictor"],
                "seed": int(seed_result["seed"]),
                "alpha": float(record["alpha"]),
                "config_id": record["config_id"],
            }
            missing_metrics = []

            for output_name, (before_name, after_name) in (
                metric_names.items()
            ):
                before = seed_metric(seed_result, before_name)
                after = seed_metric(seed_result, after_name)
                if not (np.isfinite(before) and np.isfinite(after)):
                    missing_metrics.extend(
                        name
                        for name, value in (
                            (before_name, before),
                            (after_name, after),
                        )
                        if not np.isfinite(value)
                    )
                row[f"{output_name}_before"] = before
                row[f"{output_name}_after"] = after
                row[f"{output_name}_delta"] = after - before

            if missing_metrics:
                raise AssertionError(
                    "Missing heuristic metrics for "
                    f"{record['config_id']} / seed={row['seed']} : "
                    + ", ".join(missing_metrics)
                )
            rows.append(row)

    table = pd.DataFrame(rows)
    if table.empty:
        raise ValueError(
            f"No configuration is available for {system}."
        )
    return table.sort_values(
        ["fairness_metric", "sensible_variable", "dataset", "seed"]
    ).reset_index(drop=True)


def summarize_heuristic_metrics(seed_table):
    aggregations = {"n_seeds": ("seed", "nunique")}
    for metric in HEURISTIC_METRICS:
        for stage in ("before", "after", "delta"):
            source = f"{metric}_{stage}"
            aggregations[f"{source}_mean"] = (source, "mean")
            aggregations[f"{source}_std"] = (source, "std")

    summary = (
        seed_table.groupby(
            [
                "system",
                "dataset",
                "fairness_metric",
                "sensible_variable",
                "predictor",
                "config_id",
            ],
            as_index=False,
        )
        .agg(**aggregations)
        .sort_values(
            ["fairness_metric", "sensible_variable", "dataset"]
        )
        .reset_index(drop=True)
    )
    std_columns = [
        column
        for column in summary.columns
        if column.endswith("_std")
    ]
    summary[std_columns] = summary[std_columns].fillna(0.0)
    return summary


def plot_heuristic_impact(
    dataframe,
    fairness_metric,
    sensible_variable,
    show_title=False,
):
    data = dataframe[
        (dataframe["fairness_metric"] == fairness_metric)
        & (
            dataframe["sensible_variable"]
            == sensible_variable
        )
    ].copy()
    if data.empty:
        raise ValueError(
            "No heuristic-impact data for "
            f"{fairness_metric}/{sensible_variable}."
        )

    dataset_order = [
        dataset for dataset in DATASETS
        if dataset in set(data["dataset"])
    ]
    data["dataset"] = pd.Categorical(
        data["dataset"],
        categories=dataset_order,
        ordered=True,
    )
    data = data.sort_values("dataset").reset_index(drop=True)

    fairness_label = fairness_display(fairness_metric)
    metric_specs = (
        ("risk_test", "Test error"),
        ("fairness_test", f"Test {fairness_label}"),
        ("risk_bound", "Risk upper bound"),
        ("fairness_bound", f"{fairness_label} upper bound"),
    )
    y_positions = np.arange(len(data))
    figure, axes = plt.subplots(
        1,
        len(metric_specs),
        figsize=(10.8, max(3.4, 0.42 * len(data) + 1.2)),
        sharey=True,
    )

    for axis, (metric, panel_title) in zip(axes, metric_specs):
        before = data[f"{metric}_before_mean"].to_numpy(float)
        after = data[f"{metric}_after_mean"].to_numpy(float)
        delta = data[f"{metric}_delta_mean"].to_numpy(float)

        for index, y_position in enumerate(y_positions):
            if index % 2:
                axis.axhspan(
                    y_position - 0.5,
                    y_position + 0.5,
                    color="#F5F6F8",
                    zorder=0,
                )
            change = (
                "unchanged"
                if np.isclose(delta[index], 0.0)
                else "improved"
                if delta[index] < 0.0
                else "degraded"
            )
            color = HEURISTIC_COLORS[change]
            axis.plot(
                [before[index], after[index]],
                [y_position, y_position],
                color=color,
                linestyle=HEURISTIC_LINESTYLES[change],
                linewidth=2.0,
                solid_capstyle="round",
                zorder=2,
            )
            axis.scatter(
                before[index],
                y_position,
                facecolor="white",
                edgecolor=HEURISTIC_COLORS["before"],
                linewidth=1.4,
                s=48,
                zorder=3,
            )
            axis.scatter(
                after[index],
                y_position,
                facecolor=color,
                edgecolor=color,
                linewidth=0.8,
                s=48,
                zorder=4,
            )

        values = np.concatenate([before, after])
        value_range = np.ptp(values)
        padding = max(0.01, 0.08 * value_range)
        axis.set_xlim(
            max(0.0, values.min() - padding),
            values.max() + padding,
        )
        axis.set_title(panel_title, pad=8)
        axis.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        axis.locator_params(axis="x", nbins=4)
        axis.grid(
            axis="x",
            linestyle="--",
            linewidth=0.6,
            alpha=0.7,
        )
        axis.spines["left"].set_visible(False)
        axis.tick_params(axis="y", length=0)

    axes[0].set_yticks(y_positions)
    axes[0].set_yticklabels(data["dataset"].astype(str))
    axes[0].invert_yaxis()
    axes[0].set_ylabel("Dataset")

    legend_handles = [
        Line2D(
            [0], [0],
            marker="o",
            linestyle="None",
            markerfacecolor="white",
            markeredgecolor=HEURISTIC_COLORS["before"],
            markeredgewidth=1.4,
            label="Before heuristic",
        ),
        Line2D(
            [0], [0],
            color=HEURISTIC_COLORS["improved"],
            linestyle=HEURISTIC_LINESTYLES["improved"],
            linewidth=2.0,
            label="Decrease after heuristic (better)",
        ),
        Line2D(
            [0], [0],
            color=HEURISTIC_COLORS["degraded"],
            linestyle=HEURISTIC_LINESTYLES["degraded"],
            linewidth=2.0,
            label="Increase after heuristic (worse)",
        ),
        Line2D(
            [0], [0],
            marker="o",
            color=HEURISTIC_COLORS["unchanged"],
            linestyle="None",
            label="No change",
        ),
    ]
    figure.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.01),
        ncol=4,
        frameon=False,
        handlelength=3.0,
        columnspacing=1.2,
    )
    if show_title:
        figure.suptitle(
            "Effect of the deterministic heuristic — "
            f"{fairness_label} — "
            f"{sensitive_display(data['dataset'].iloc[0], sensible_variable)}"
        )
    figure.tight_layout(rect=(0.0, 0.10, 1.0, 1.0))
    return figure, data


heuristic_seed_metrics = build_heuristic_seed_metrics(
    selection_records
)
heuristic_task_summary = summarize_heuristic_metrics(
    heuristic_seed_metrics
)
heuristic_artifacts = {}

for fairness_metric in PLOT_FAIRNESS_METRICS:
    for sensible_variable in SENSITIVE_VARIABLES:
        figure, plot_data = plot_heuristic_impact(
            heuristic_task_summary,
            fairness_metric,
            sensible_variable,
            show_title=False,
        )
        heuristic_artifacts[
            (fairness_metric, sensible_variable)
        ] = (figure, plot_data)
        plt.show()

## 10. Empirical trade-off by dataset

For each dataset, test fairness violation is shown on the x-axis and
minority-class test F1 on the y-axis. The large marker is the five-seed mean,
and error bars show one standard deviation. The arrow points toward the ideal
region: low violation and high F1.


In [ ]:
SYSTEM_MARKERS = {
    "ours_det_rf": "o",
    "ours_sto_rf": "s",
    "ferm_linear": "D",
    "reduction_trees": "P",
}


def plot_empirical_tradeoff_by_dataset(
    seed_data,
    fairness_metric,
    sensible_variable,
    y_metric=EMPIRICAL_Y_METRIC,
    show_ideal_arrow=SHOW_EMPIRICAL_IDEAL_ARROW,
    systems=EMPIRICAL_METHODS,
    show_seed_points=False,
):
    metric_specs = {
        "risk": {
            "column": "risk_test",
            "axis_label": "Test error",
        },
        "fscore": {
            "column": "f1_test",
            "axis_label": "F1-score",
        },
    }
    if y_metric not in metric_specs:
        raise ValueError("y_metric must be 'risk' or 'fscore'.")
    y_spec = metric_specs[y_metric]

    data = seed_data[
        seed_data["system"].isin(systems)
        & (seed_data["fairness_metric"] == fairness_metric)
        & (seed_data["sensible_variable"] == sensible_variable)
    ].copy()
    if data.empty:
        raise ValueError(
            "No empirical data for "
            f"{fairness_metric}/{sensible_variable}."
        )

    dataset_order = [
        dataset for dataset in DATASETS
        if dataset in set(data["dataset"])
    ]
    figure, axes = plt.subplots(
        1,
        len(dataset_order),
        figsize=(1.35 * len(dataset_order), 2.45),
        squeeze=False,
    )
    axes = axes.ravel()

    for axis_index, (axis, dataset) in enumerate(zip(axes, dataset_order)):
        axis.set_box_aspect(1.0)
        panel = data[data["dataset"] == dataset]
        available_systems = [
            system for system in systems
            if system in set(panel["system"])
        ]

        for system in available_systems:
            method_data = panel[
                panel["system"] == system
            ].replace([np.inf, -np.inf], np.nan)
            method_data = method_data.dropna(
                subset=["fairness_test", y_spec["column"]]
            )
            if method_data.empty:
                continue

            fairness_values = method_data["fairness_test"].to_numpy(
                dtype=float
            )
            y_values = method_data[y_spec["column"]].to_numpy(dtype=float)
            color = SYSTEM_COLORS[system]
            marker = SYSTEM_MARKERS[system]

            if show_seed_points:
                axis.scatter(
                    fairness_values,
                    y_values,
                    s=22,
                    marker=marker,
                    facecolor=color,
                    edgecolor="none",
                    alpha=0.25,
                    zorder=2,
                )

            fairness_mean = float(np.mean(fairness_values))
            y_mean = float(np.mean(y_values))
            fairness_std = (
                float(np.std(fairness_values, ddof=1))
                if len(fairness_values) > 1
                else 0.0
            )
            y_std = (
                float(np.std(y_values, ddof=1))
                if len(y_values) > 1
                else 0.0
            )

            axis.errorbar(
                fairness_mean,
                y_mean,
                xerr=fairness_std,
                yerr=y_std,
                fmt="none",
                ecolor=color,
                elinewidth=1.0,
                capsize=2.5,
                alpha=0.85,
                zorder=3,
            )
            axis.scatter(
                fairness_mean,
                y_mean,
                s=35,
                marker=marker,
                facecolor=color,
                edgecolor="black",
                linewidth=0.8,
                zorder=4,
            )

        axis.set_title(
            f"{dataset} - {sensitive_display(dataset, sensible_variable)}",
            fontsize=7,
            fontweight="bold",
        )
        axis.set_xlabel(
            f"{fairness_display(fairness_metric)} test violation",
            fontsize=9,
        )
        if axis_index == 0:
            axis.set_ylabel(y_spec["axis_label"], fontsize=9)

        axis.tick_params(axis="both", labelsize=8)
        axis.locator_params(axis="x", nbins=3)
        axis.locator_params(axis="y", nbins=4)
        axis.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        axis.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        clean_axis(axis)
        axis.grid(axis="both", alpha=0.22, linewidth=0.6)
        axis.margins(x=0.1, y=0.1)

        if show_ideal_arrow and y_metric == "fscore":
            axis.text(
                0.10,
                0.90,
                r"$\nwarrow$",
                transform=axis.transAxes,
                color="#E69F00",
                fontsize=10,
                ha="center",
                va="center",
                zorder=6,
            )

    method_handles = [
        Line2D(
            [0],
            [0],
            marker=SYSTEM_MARKERS[system],
            color="none",
            markerfacecolor=SYSTEM_COLORS[system],
            markeredgecolor="black",
            markersize=7,
            label=SYSTEM_LABELS[system],
        )
        for system in systems
        if system in set(data["system"])
    ]
    figure.legend(
        handles=method_handles,
        bbox_to_anchor=(0.01, 0.8),
        loc="upper left",
        ncol=len(method_handles),
        fontsize=8,
        frameon=False,
        borderaxespad=0.0,
        handlelength=1.6,
        handletextpad=0.45,
        columnspacing=1.1,
    )
    figure.tight_layout(rect=(0.0, 0.0, 1.0, 0.86))
    figure.subplots_adjust(wspace=0.35)
    return figure, data


empirical_artifacts = {}
for fairness_metric in PLOT_FAIRNESS_METRICS:
    for sensible_variable in SENSITIVE_VARIABLES:
        empirical_artifacts[(fairness_metric, sensible_variable)] = (
            plot_empirical_tradeoff_by_dataset(
                selected_seeds,
                fairness_metric,
                sensible_variable,
                y_metric=EMPIRICAL_Y_METRIC,
                show_seed_points=False,
            )
        )


empirical_tradeoff_test_tables = {}
for fairness_metric in PLOT_FAIRNESS_METRICS:
    table = (
        selected_seeds[
            selected_seeds["system"].isin(EMPIRICAL_METHODS)
            & (selected_seeds["fairness_metric"] == fairness_metric)
        ]
        .groupby(
            [
                "dataset",
                "sensible_variable",
                "system",
                "system_label",
            ],
            as_index=False,
        )
        .agg(
            n_seeds=("seed", "nunique"),
            risk_test_mean=("risk_test", "mean"),
            risk_test_std=("risk_test", "std"),
            fairness_test_mean=("fairness_test", "mean"),
            fairness_test_std=("fairness_test", "std"),
            fscore_test_mean=("f1_test", "mean"),
            fscore_test_std=("f1_test", "std"),
        )
        .sort_values(["dataset", "sensible_variable", "system"])
        .reset_index(drop=True)
    )
    std_columns = [
        "risk_test_std",
        "fairness_test_std",
        "fscore_test_std",
    ]
    table[std_columns] = table[std_columns].fillna(0.0)
    empirical_tradeoff_test_tables[fairness_metric] = table
    display(Markdown(f"### Test results — {fairness_display(fairness_metric)}"))


## 11. LaTeX tables for the paper

The five tables are generated when `EXPORT_LATEX_TABLES=True`.


In [ ]:
LATEX_TABLE_KEY_COLUMNS = [
    "system",
    "dataset",
    "fairness_metric",
    "sensible_variable",
]


def aggregate_seed_table(systems, metric_keys):
    aggregations = {}
    for metric in metric_keys:
        aggregations[f"{metric}_mean"] = (metric, "mean")
        aggregations[f"{metric}_std"] = (metric, "std")
    return (
        selected_seeds[
            selected_seeds["system"].isin(systems)
        ]
        .groupby(LATEX_TABLE_KEY_COLUMNS, as_index=False)
        .agg(**aggregations)
    )


def latex_escape(value):
    bs = chr(92)
    text = str(value)
    for character in ("&", "%", "_", "#"):
        text = text.replace(character, f"{bs}{character}")
    return text


def latex_metric_spec(
    key,
    label,
    best="min",
    scale=100.0,
    decimals=2,
    show_std=True,
):
    return {
        "key": key,
        "label": label,
        "best": best,
        "scale": scale,
        "decimals": decimals,
        "show_std": show_std,
    }


def latex_metric_groups(
    metric_specs,
    fairness_metrics=FAIRNESS_METRICS,
):
    return [
        (
            fairness_display(fairness_metric),
            [
                latex_metric_spec(
                    label=(
                        fairness_display(fairness_metric)
                        if label == "FAIRNESS"
                        else label
                    ),
                    **spec,
                )
                for label, spec in metric_specs
            ],
        )
        for fairness_metric in fairness_metrics
    ]


def latex_table_cell(
    stats_index,
    systems,
    dataset,
    sensible_variable,
    fairness_metric,
    system,
    spec,
):
    key = (
        system,
        dataset,
        fairness_metric,
        sensible_variable,
    )
    if key not in stats_index.index:
        return "N/A"

    row = stats_index.loc[key]
    mean = row.get(f"{spec['key']}_mean", np.nan)
    if not np.isfinite(mean):
        return "N/A"

    displayed_values = []
    for candidate_system in systems:
        candidate_key = (
            candidate_system,
            dataset,
            fairness_metric,
            sensible_variable,
        )
        if candidate_key not in stats_index.index:
            continue
        candidate = stats_index.loc[candidate_key].get(
            f"{spec['key']}_mean",
            np.nan,
        )
        if np.isfinite(candidate):
            displayed_values.append(
                float(
                    f"{candidate * spec['scale']:.{spec['decimals']}f}"
                )
            )

    displayed_mean = float(
        f"{mean * spec['scale']:.{spec['decimals']}f}"
    )
    if spec["best"] == "max":
        best_displayed = max(displayed_values)
    else:
        best_displayed = min(displayed_values)
    is_best = displayed_mean == best_displayed

    mean_text = f"{displayed_mean:.{spec['decimals']}f}"
    std = row.get(f"{spec['key']}_std", np.nan)
    if spec["show_std"] and np.isfinite(std):
        std_text = (
            f"{std * spec['scale']:.{spec['decimals']}f}"
        )
        cell = mean_text + " $" + chr(92) + "pm$ " + std_text
    else:
        cell = mean_text
    if is_best:
        cell = f"{chr(92)}textbf{{{cell}}}"
    return cell


def build_latex_results_table(
    stats,
    systems,
    metric_groups,
    caption,
    label,
    fairness_metrics=FAIRNESS_METRICS,
):
    bs = chr(92)
    stats_index = stats.set_index(LATEX_TABLE_KEY_COLUMNS)
    group_widths = [len(specs) for _, specs in metric_groups]
    n_columns = 3 + sum(group_widths)
    alignment = (
        "|ccc|"
        + "|".join("c" * width for width in group_widths)
        + "|"
    )
    lines = [
        f"{bs}begin{{table}}[ht]",
        f"{bs}scriptsize",
        f"{bs}centering",
        f"{bs}caption{{{caption}}}",
        f"{bs}label{{{label}}}",
        f"{bs}resizebox{{{bs}linewidth}}{{!}}{{%",
        f"{bs}begin{{tabular}}{{{alignment}}}",
        f"{bs}hline",
    ]

    top_header = [
        f"{bs}multirow{{2}}{{*}}{{Dataset}}",
        f"{bs}multirow{{2}}{{*}}{{Sens. attr.}}",
        f"{bs}multirow{{2}}{{*}}{{Type}}",
    ]
    top_header.extend(
        f"{bs}multicolumn{{{len(specs)}}}{{c|}}{{{group_label}}}"
        for group_label, specs in metric_groups
    )
    lines.append(" & ".join(top_header) + f" {bs}{bs}")
    lines.append(f"{bs}cline{{4-{n_columns}}}")

    second_header = ["", "", ""]
    second_header.extend(
        latex_escape(spec["label"])
        for _, specs in metric_groups
        for spec in specs
    )
    lines.append(" & ".join(second_header) + f" {bs}{bs}")
    lines.append(f"{bs}hline")

    dataset_span = len(SENSITIVE_VARIABLES) * len(systems)
    for dataset_index, dataset in enumerate(DATASETS):
        for sensitive_index, sensible_variable in enumerate(
            SENSITIVE_VARIABLES
        ):
            for system_index, system in enumerate(systems):
                row = []
                if sensitive_index == 0 and system_index == 0:
                    row.append(
                        f"{bs}multirow{{{dataset_span}}}{{*}}"
                        f"{{{latex_escape(dataset)}}}"
                    )
                else:
                    row.append("")
                if system_index == 0:
                    row.append(
                        f"{bs}multirow{{{len(systems)}}}{{*}}"
                        f"{{{latex_escape(sensitive_display(dataset, sensible_variable))}}}"
                    )
                else:
                    row.append("")
                row.append(latex_escape(SYSTEM_LABELS[system]))

                for fairness_metric, (_, specs) in zip(
                    fairness_metrics,
                    metric_groups,
                ):
                    for spec in specs:
                        row.append(
                            latex_table_cell(
                                stats_index,
                                systems,
                                dataset,
                                sensible_variable,
                                fairness_metric,
                                system,
                                spec,
                            )
                        )
                lines.append(" & ".join(row) + f" {bs}{bs}")

            if sensitive_index < len(SENSITIVE_VARIABLES) - 1:
                lines.append(f"{bs}cline{{2-{n_columns}}}")
            else:
                lines.append(f"{bs}hline")

    lines.extend(
        [
            f"{bs}end{{tabular}}%",
            "}",
            f"{bs}end{{table}}",
        ]
    )
    return "\n".join(lines)


bound_seed_stats = aggregate_seed_table(
    BOUND_METHODS,
    ("risk_bound", "fairness_bound", "risk_test", "fairness_test"),
)
empirical_seed_stats = aggregate_seed_table(
    EMPIRICAL_METHODS,
    ("risk_test", "fairness_test", "f1_test"),
)
bound_metric_groups = latex_metric_groups(
    (
        ("Risk", {"key": "risk_bound"}),
        ("FAIRNESS", {"key": "fairness_bound"}),
    )
)
bound_empirical_metric_groups = latex_metric_groups(
    (
        ("Risk", {"key": "risk_test"}),
        ("FAIRNESS", {"key": "fairness_test"}),
    )
)
empirical_metric_specs = (
    ("Risk", {"key": "risk_test"}),
    ("FAIRNESS", {"key": "fairness_test"}),
    ("F1", {"key": "f1_test", "best": "max"}),
)
LATEX_TABLES = {
    "bound_values": build_latex_results_table(
        bound_seed_stats,
        BOUND_METHODS,
        bound_metric_groups,
        "Average risk and fairness bounds over the available runs.",
        "tab:bound-values",
    ),
    "bound_empirical_values": build_latex_results_table(
        bound_seed_stats,
        BOUND_METHODS,
        bound_empirical_metric_groups,
        "Average empirical test risk and fairness for bound-producing methods.",
        "tab:bound-empirical-values",
    ),
    **{
        f"empirical_all_methods_{fairness_metric.lower()}": (
            build_latex_results_table(
                empirical_seed_stats[
                    empirical_seed_stats["fairness_metric"]
                    == fairness_metric
                ],
                EMPIRICAL_METHODS,
                latex_metric_groups(
                    empirical_metric_specs,
                    fairness_metrics=(fairness_metric,),
                ),
                (
                    "Average empirical test risk, fairness, "
                    "and F1-score for "
                    f"{fairness_display(fairness_metric)}."
                ),
                (
                    "tab:empirical-all-methods-"
                    f"{fairness_metric.lower()}"
                ),
                fairness_metrics=(fairness_metric,),
            )
        )
        for fairness_metric in FAIRNESS_METRICS
    },

}

for table_name, latex_source in LATEX_TABLES.items():
    display(Markdown(f"### {table_name}.tex"))
    display(Code(latex_source, language="latex"))


## 12. Audits and exports


In [ ]:
bound_results_table = task_summary[
    task_summary["system"].isin(BOUND_METHODS)
][
    [
        "system_label",
        "dataset",
        "fairness_metric",
        "sensible_variable",
        "risk_bound",
        "risk_test",
        "fairness_bound",
        "fairness_test",
        "f1_test",
    ]
].reset_index(drop=True)


def expected_task_keys(system):
    spec = SYSTEM_SPECS[system]
    return {
        (dataset, fairness_metric, sensible_variable)
        for dataset in DATASETS
        for fairness_metric in FAIRNESS_METRICS
        if fairness_metric in spec["supported_metrics"]
        for sensible_variable in SENSITIVE_VARIABLES
    }


def run_smoke_tests():
    assert set(selected_seeds["dataset"]) == set(DATASETS)
    assert set(selected_seeds["seed"]) == set(PAPER_SEEDS)
    assert not selected_configs.duplicated(
        ["system", *TASK_KEYS]
    ).any()
    assert not selected_seeds.duplicated(
        ["system", *TASK_KEYS, "seed"]
    ).any()

    for system in ACTIVE_SYSTEMS:
        observed = set(
            selected_configs.loc[
                selected_configs["system"] == system,
                TASK_KEYS,
            ].itertuples(index=False, name=None)
        )
        expected = expected_task_keys(system)
        assert observed == expected, (
            system,
            len(expected - observed),
            len(observed - expected),
        )

    seed_counts = (
        selected_seeds.groupby(["system", *TASK_KEYS])["seed"]
        .nunique()
    )
    assert seed_counts.eq(len(PAPER_SEEDS)).all()

    selection_metrics = (
        selected_configs["selection_risk_metric"].fillna("")
        + " "
        + selected_configs["selection_fairness_metric"].fillna("")
    )
    assert not selection_metrics.str.contains("_te_").any()

    pb_rows = selected_seeds[selected_seeds["family"] == "pac_bayes"]
    assert set(pb_rows["f1_source"]) == {
        "reconstructed_from_class_errors"
    }
    assert pb_rows["f1_train"].notna().all()
    assert pb_rows["f1_test"].notna().all()

    external_rows = selected_seeds[
        selected_seeds["family"].isin({"ferm", "reduction"})
    ]
    assert set(external_rows["f1_source"]) == {"stored_empirical"}
    assert external_rows[["risk_bound", "fairness_bound"]].isna().all().all()

    for column in ("risk_test", "fairness_test", "f1_test"):
        values = selected_seeds[column].dropna()
        assert values.between(-1e-9, 1.0 + 1e-9).all()

    heuristic_observed = set(
        heuristic_task_summary[TASK_KEYS].itertuples(
            index=False,
            name=None,
        )
    )
    assert heuristic_observed == expected_task_keys(HEURISTIC_SYSTEM)
    assert not heuristic_seed_metrics.duplicated(
        [*TASK_KEYS, "seed"]
    ).any()
    heuristic_values = heuristic_seed_metrics.filter(
        regex=r"_(before|after|delta)$"
    ).to_numpy(dtype=float)
    assert np.isfinite(heuristic_values).all()

    assert np.isclose(
        fscore_from_errors(0.0, 0.0, 10, 20, -1),
        1.0,
    )
    assert len(LATEX_TABLES) == 5

    return {
        "datasets": list(DATASETS),
        "seeds": list(PAPER_SEEDS),
        "systems": list(ACTIVE_SYSTEMS),
        "selected_configs": len(selected_configs),
        "selected_seed_rows": len(selected_seeds),
        "heuristic_seed_rows": len(heuristic_seed_metrics),
        "heuristic_tasks": len(heuristic_task_summary),
        "minimum_seeds_per_task": int(seed_counts.min()),
        "maximum_seeds_per_task": int(seed_counts.max()),
        "pac_bayes_f1_source": "reconstructed_from_class_errors",
        "zero_f1_tasks": int(task_summary["zero_f1_all_seeds"].sum()),
        "constant_classifier_tasks": int(
            task_summary["constant_all_seeds"].sum()
        ),
    }


smoke_test_summary = run_smoke_tests()
display(pd.Series(smoke_test_summary, name="value").to_frame())


ARTIFACTS = {
    "tables": {
        **{
            f"empirical_tradeoff_test_{metric.lower()}": table
            for metric, table in empirical_tradeoff_test_tables.items()
        },
        "selected_configs": selected_configs,
        "selected_seed_metrics": selected_seeds,
        "heuristic_before_after_by_seed": heuristic_seed_metrics,
        "heuristic_before_after_summary": heuristic_task_summary,
        "bound_results": bound_results_table,
    },
    "latex_tables": LATEX_TABLES,
    "figures": {},
}

for (fairness_metric, sensible_variable), (figure, _) in (
    bound_artifacts.items()
):
    ARTIFACTS["figures"][
        f"bounds_{fairness_metric}_{sensible_variable}".lower()
    ] = figure

for (fairness_metric, sensible_variable), (figure, _) in (
    heuristic_artifacts.items()
):
    ARTIFACTS["figures"][
        f"heuristic_{fairness_metric}_{sensible_variable}".lower()
    ] = figure

for (fairness_metric, sensible_variable), (figure, _) in (
    empirical_artifacts.items()
):
    ARTIFACTS["figures"][
        f"empirical_{fairness_metric}_{sensible_variable}".lower()
    ] = figure


def export_artifacts(artifacts, output_dir=OUTPUT_DIR):
    data_dir = output_dir / "data"
    figure_dir = output_dir / "figures"
    latex_dir = output_dir / "tables"
    figure_dir.mkdir(parents=True, exist_ok=True)

    if EXPORT_TABLES:
        data_dir.mkdir(parents=True, exist_ok=True)
        for name, table in artifacts["tables"].items():
            table.to_csv(data_dir / f"{name}.csv", index=False)

    exported_latex_tables = []
    if EXPORT_LATEX_TABLES:
        latex_dir.mkdir(parents=True, exist_ok=True)
        for name, latex_source in artifacts["latex_tables"].items():
            output_path = latex_dir / f"{name}.tex"
            output_path.write_text(latex_source + "\n", encoding="utf-8")
            exported_latex_tables.append(
                str(output_path.relative_to(output_dir))
            )

    exported_figures = []
    for name, figure in artifacts["figures"].items():
        family = name.split("_", 1)[0]
        family_dir = figure_dir / family
        family_dir.mkdir(parents=True, exist_ok=True)
        output_path = family_dir / f"{name}.pdf"
        figure.savefig(
            output_path,
            format="pdf",
            bbox_inches="tight",
            pad_inches=EXPORT_PDF_PAD_INCHES,
            facecolor="white",
            transparent=False,
            metadata={
                "Title": name.replace("_", " "),
                "Subject": "PBFAIR benchmark results",
            },
        )
        exported_figures.append(str(output_path.relative_to(output_dir)))

    manifest = {
        "datasets": list(DATASETS),
        "seeds": list(PAPER_SEEDS),
        "fairness_metrics": list(FAIRNESS_METRICS),
        "plot_fairness_metrics": list(PLOT_FAIRNESS_METRICS),
        "sensitive_variables": list(SENSITIVE_VARIABLES),
        "bound_methods": list(BOUND_METHODS),
        "empirical_methods": list(EMPIRICAL_METHODS),
        "heuristic_system": HEURISTIC_SYSTEM,
        "empirical_y_metric": EMPIRICAL_Y_METRIC,
        "figure_formats": ["pdf"],
        "pdf_fonttype": 42,
        "pdf_pad_inches": EXPORT_PDF_PAD_INCHES,
        "tables_exported": EXPORT_TABLES,
        "latex_tables_exported": EXPORT_LATEX_TABLES,
        "latex_table_files": exported_latex_tables,
        "figure_files": exported_figures,
        "f1_pac_bayes": "reconstructed_from_class_errors",
        "table_shapes": {
            name: list(table.shape)
            for name, table in artifacts["tables"].items()
        },
    }
    (output_dir / "manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    return manifest


manifest = None
if SAVE_EXPORTS:
    manifest = export_artifacts(ARTIFACTS)
    print(
        f"{len(manifest['figure_files'])} PDF figures "
        f"written to {OUTPUT_DIR / 'figures'}"
    )
    print(
        f"{len(manifest['latex_table_files'])} LaTeX tables "
        f"written to {OUTPUT_DIR / 'tables'}"
    )
else:
    print("SAVE_EXPORTS=False: no file exported.")

print(
    "Notebook ready. PAC-Bayes F1 was reconstructed "
    "for every selected row."
)
